# Semantic scoring on Colab GPU (team metric standard)

Runs `scripts/score_semantic.py` (rescaled BERTScore + length-weighted MPNet cosine) for the full Ship of Theseus analysis on a Colab GPU, instead of about 8 hours on a laptop. Everything else (lexical scores, style features, the analysis notebook) stays local.

**Before running**
1. On the Mac, run `notebooks/sot_analysis.ipynb` §1–§2 up to the pairs cell (it writes `data/derived/sot_sample.parquet` and `sot_floors.parquet`), then build the bundle:
   ```bash
   zip -r -9 data/colab/sot_colab_bundle.zip src scripts config.yaml pyproject.toml \
       data/sot/paraphrased_datasets data/derived/sot_sample.parquet data/derived/sot_floors.parquet
   ```
2. Upload `sot_colab_bundle.zip` to Google Drive at **`MyDrive/sot_colab/`**.
3. In Colab: **Runtime → Change runtime type → A100 GPU** (L4 also works, just slower), then run all cells.

**Outputs** go to `MyDrive/sot_colab/derived/`: `sot_pairs_semantic_v2.parquet`, `sot_floors_semantic_v2.parquet` and `semantic_run.json` (the precision and GPU used). Download them into the repo's `data/derived/`, then continue the analysis notebook from §2.

**Disconnects:** the caches live on Drive and are written atomically, so if the runtime drops, reconnect and run all cells again. Originals already scored are skipped.

In [ ]:
# 1. Drive, GPU and paths
import json, shutil, subprocess, sys, time
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

DRIVE = Path("/content/drive/MyDrive/sot_colab")
BUNDLE = DRIVE / "sot_colab_bundle.zip"
DERIVED = DRIVE / "derived"   # caches persist on Drive across disconnects
WORK = Path("/content/work")  # code + corpus unpacked on the fast local disk
assert BUNDLE.exists(), f"upload sot_colab_bundle.zip to {DRIVE} first"
DERIVED.mkdir(parents=True, exist_ok=True)

import torch
assert torch.cuda.is_available(), "no GPU: Runtime -> Change runtime type -> A100 GPU"
GPU = torch.cuda.get_device_name(0)
GPU_MEM = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"GPU: {GPU} ({GPU_MEM:.0f} GB) | torch {torch.__version__}")

In [ ]:
# 2. Unpack the bundle and install the one missing dependency
if not (WORK / "scripts/score_semantic.py").exists():
    WORK.mkdir(parents=True, exist_ok=True)
    shutil.unpack_archive(BUNDLE, WORK)
for name in ("sot_sample.parquet", "sot_floors.parquet"):   # the inputs the script reads, next to its caches
    if not (DERIVED / name).exists():
        shutil.copy2(WORK / "data/derived" / name, DERIVED / name)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bert-score==0.3.13"], check=True)

ENV = {"PYTHONPATH": str(WORK / "src"), "TOKENIZERS_PARALLELISM": "false"}
# Batch settings by GPU memory: 40-80 GB (A100) vs 24 GB (L4) vs 16 GB (T4/V100).
BATCH = {"jobs_per_batch": 32, "batch_size": 64} if GPU_MEM > 30 else \
        {"jobs_per_batch": 16, "batch_size": 32} if GPU_MEM > 20 else {"jobs_per_batch": 8, "batch_size": 16}


def score(*extra, log="semantic.log"):
    """Run scripts/score_semantic.py with the Drive caches; streams the log and appends it to Drive."""
    import os
    cmd = [sys.executable, "scripts/score_semantic.py", "--derived", str(DERIVED),
           "--jobs-per-batch", str(BATCH["jobs_per_batch"]), "--batch-size", str(BATCH["batch_size"]), *extra]
    print(" ".join(cmd[1:]), flush=True)
    with open(DERIVED / log, "a") as f, subprocess.Popen(
            cmd, cwd=WORK, env={**os.environ, **ENV}, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as p:
        for line in p.stdout:
            if any(k in line for k in ("jobs,", "done", "originals", "Error", "Traceback", "error")):
                print(line, end="", flush=True)
            f.write(line)
    if p.returncode:
        raise RuntimeError(f"score_semantic.py failed (exit {p.returncode}); see {DERIVED / log}")


print("bundle unpacked; batch settings:", BATCH)

## fp16 check (about 2-3 minutes)

fp16 is several times faster on A100/L4 tensor cores, but its scores differ slightly from fp32, and fp32 is the team reference. This cell scores the same 100 originals both ways and compares them. fp16 is used only when every score stays within **0.002** of fp32, which is well below the width of the confidence intervals; otherwise the full run stays in fp32.

In [ ]:
# 3. Compare fp16 with fp32 on 100 originals (skipped if a full run already chose a precision)
import pandas as pd

RUN_INFO = DERIVED / "semantic_run.json"
TOLERANCE = 0.002
METRICS = ["bertscore_f1_rescaled", "bertscore_p_rescaled", "bertscore_r_rescaled", "mpnet_cosine"]
KEYS = ["dataset", "source", "key", "variant", "round", "kind"]

if RUN_INFO.exists():   # resuming: never mix precisions inside one cache
    DTYPE = json.loads(RUN_INFO.read_text())["dtype"]
    print(f"resuming the earlier run in {DTYPE}")
else:
    timing = {}
    for dt in ("fp32", "fp16"):
        for f in DERIVED.glob(f"sot_pairs_semantic_v2_check_{dt}*"):
            f.unlink()  # always a fresh comparison
        t = time.time()
        score("--limit", "100", "--dtype", dt, "--suffix", f"_check_{dt}", log="check.log")
        timing[dt] = time.time() - t
    a = pd.read_parquet(DERIVED / "sot_pairs_semantic_v2_check_fp32.parquet")
    b = pd.read_parquet(DERIVED / "sot_pairs_semantic_v2_check_fp16.parquet")
    m = a.merge(b, on=KEYS, suffixes=("_32", "_16"), validate="1:1")
    diff = pd.Series({c: (m[f"{c}_32"] - m[f"{c}_16"]).abs().max() for c in METRICS})
    print(f"{len(m):,} pairs | time fp32 {timing['fp32']:.0f}s, fp16 {timing['fp16']:.0f}s (both include model loading)")
    print("max |fp32 - fp16|:", diff.round(5).to_dict())
    DTYPE = "fp16" if diff.max() <= TOLERANCE else "fp32"
    print(f"-> full run in {DTYPE}")
    RUN_INFO.write_text(json.dumps({"dtype": DTYPE, "gpu": GPU, "max_abs_diff_fp16_vs_fp32": diff.round(6).to_dict(),
                                    "tolerance": TOLERANCE, **BATCH, "started": time.strftime("%Y-%m-%d %H:%M")}, indent=1))

## Full run

This scores 16,682 originals (583,870 pairs), then the 1,400 floor pairs. Progress lines appear every 200 originals. If the runtime disconnects, reconnect and **run all cells again**: it resumes from the Drive caches in the same precision.

In [ ]:
# 4. Full run (resumable)
t = time.time()
score("--dtype", DTYPE)
info = json.loads(RUN_INFO.read_text())
info.update(finished=time.strftime("%Y-%m-%d %H:%M"), last_session_minutes=round((time.time() - t) / 60, 1))
RUN_INFO.write_text(json.dumps(info, indent=1))

pairs_sem = pd.read_parquet(DERIVED / "sot_pairs_semantic_v2.parquet")
floors_sem = pd.read_parquet(DERIVED / "sot_floors_semantic_v2.parquet")
sample = pd.read_parquet(DERIVED / "sot_sample.parquet")
n_orig = pairs_sem[["dataset", "source", "key"]].drop_duplicates().shape[0]
print(f"pairs: {len(pairs_sem):,} rows for {n_orig:,} / {len(sample):,} originals | floors: {len(floors_sem):,} / 1,400")
assert n_orig == len(sample) and len(floors_sem) == 1400, "incomplete: run this cell again to resume"
print("Done. Download sot_pairs_semantic_v2.parquet, sot_floors_semantic_v2.parquet and semantic_run.json "
      f"from {DERIVED} into the repo's data/derived/.")